# 03: Build a quantum training loop

You will define a one-qubit model, choose a target, inspect its gradient, and update its parameter yourself.
Then you will repeat that update with a short PyTorch loop.
No training helper is called: each part of the algorithm is in this notebook.

## 1. Define a circuit with an adjustable angle

The RY gate rotates a qubit by an angle `theta`. Our function describes how to build the circuit from the current parameter.
The optional `inputs` argument is unused in this small example; it can carry data in a larger model.

In [ ]:
import flagquantum as fq
import torch


def build_circuit(parameters, inputs=None):
    circuit = fq.Circuit(1)
    circuit.ry(0, parameters["theta"][0])
    return circuit


## 2. Make the angle trainable

`fq.Module` owns a trainable PyTorch parameter named `theta`. The shape `(1,)` means one number.
The seed gives us the same starting point when we restart this cell.

We ask the model to return the Z expectation of qubit 0. A Z measurement assigns +1 to `0` and -1 to `1`; its expectation is their average.
For this circuit, the answer is `cos(theta)`.

In [ ]:
policy = fq.RuntimePolicy(
    execution_options=fq.ExecutionOptions(device="cpu", mode="statevector"),
    observable="z_sum",
    observable_wires=(0,),
)
model = fq.Module(
    build_circuit,
    parameters={"theta": (1,)},
    init="uniform",
    seed=42,
    policy=policy,
)
for name, parameter in model.named_parameters():
    print(name, parameter.detach())
print("Initial prediction:", model().detach())


## 3. Choose what the model should learn

The target is 0.25. The loss is the squared difference between the prediction and the target.
**Before running:** what would a loss of zero tell you?

In [ ]:
target = 0.25
prediction = model()
loss = (prediction - target).square().mean()
initial_loss = loss.item()
print("Prediction:", prediction.item())
print("Target:", target)
print("Loss:", initial_loss)


## 4. Inspect the gradient

`backward()` computes how the loss changes with the angle.
The optimizer will use that gradient to adjust the parameter. Clearing old gradients matters because PyTorch accumulates them.

In [ ]:
learning_rate = 0.08
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
optimizer.zero_grad()
loss.backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None
    assert torch.isfinite(parameter.grad).all()
    print(name, "gradient:", parameter.grad)


## 5. Take one update

This is the first optimizer step. Compare the new prediction with the old one.
To repeat the whole experiment, restart from the cell that creates `model`; rerunning `backward()` on an already-used graph is not a fresh training step.

In [ ]:
losses = [initial_loss]
optimizer.step()
print("Prediction after one update:", model().detach().item())


## 6. Repeat the same process

Each iteration makes a fresh prediction, computes its loss, differentiates it, and updates the angle.
We already took one step, so 79 more gives us 80 in total.

In [ ]:
total_steps = 80
for step in range(1, total_steps):
    optimizer.zero_grad()
    prediction = model()
    loss = (prediction - target).square().mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().item())

final_prediction = model().detach().item()
final_loss = (final_prediction - target) ** 2
print("Initial loss:", initial_loss)
print("Final loss:", final_loss)
print("Final prediction:", final_prediction)
assert final_loss < 0.01


## 7. See the learning process

The curve need not decrease on every step. With the default settings, the final loss should be below 0.01.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.xlabel("Optimizer step")
plt.ylabel("Squared error")
plt.show()


## Try a change

Change `target` to another value between -1 and 1, or change `learning_rate`, then restart and run the notebook.
Keep these values in this notebook; you do not need to edit a separate Python file.
Why can this model never reach a target of 2?

For a larger example, try `examples/quick_start.py`, where one optimizer trains a classical layer and a quantum layer.
Notebook 05 is a separate Bell hardware experiment; this training loop does not automatically submit its parameter to a QPU.